# Virtual Mouse / Air Canvas - Phase 7

## Air-writing -> typed digits (reliable, no TensorFlow)

Write a digit (0-9) in the air and have it recognised and typed. This uses
**scikit-learn** instead of a deep-learning framework, on purpose:

- scikit-learn has **no protobuf dependency**, so it coexists with MediaPipe
  in your existing environment - no second venv, no ONNX bridge, no version
  conflicts (TensorFlow and MediaPipe genuinely cannot share an environment).
- For clean air-drawn digits a small scikit-learn model is plenty accurate.

Two parts:
1. **Train once** (this notebook) -> saves `digit_model.joblib`.
2. **Run the live app** (`air_writer.py`, written by this notebook) -> draw and type.

---
### Honest expectation
Air-drawn digits are messier than dataset digits, so accuracy is good but not
perfect. Draw digits large, centred, and clearly for best results.

## 1. Install (run once)
scikit-learn and joblib coexist with your existing MediaPipe install.

In [4]:
!pip install scikit-learn joblib pandas

  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ------ --------------------------------- 1.6/9.8 MB 13.9 MB/s eta 0:00:01
   -------------------- ------------------- 5.0/9.8 MB 15.1 MB/s eta 0:00:01
   ------------------------------- -------- 7.6/9.8 MB 14.2 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.8 MB 12.8 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 12.2 MB/s  0:00:00
Using cached tzdata-2026.2-py2.py3-none-any.whl (349 kB)

   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------

## 2. Train the digit model

Downloads MNIST (70,000 handwritten digits) via scikit-learn, trains a small
neural network (MLP), reports test accuracy, and saves it to
`digit_model.joblib`. The first run downloads ~15 MB and training takes a
couple of minutes on your i7. After that you never need to run this again.

In [5]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
import joblib

print("Downloading MNIST (first time only)...")
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X = mnist.data.astype(np.float32) / 255.0      # normalise 0..1
y = mnist.target.astype(int)
print("Data:", X.shape, "labels:", y.shape)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.1, random_state=42)

print("Training MLP classifier (this takes a couple of minutes)...")
clf = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=30,
                    early_stopping=True, random_state=42, verbose=False)
clf.fit(X_tr, y_tr)

acc = clf.score(X_te, y_te)
print(f"Test accuracy: {acc:.3f}")

joblib.dump(clf, "digit_model.joblib")
print("Saved digit_model.joblib - ready for the live app.")

Data: (70000, 784) labels: (70000,)
Training MLP classifier (this takes a couple of minutes)...
Test accuracy: 0.973
Saved digit_model.joblib - ready for the live app.


## 3. Sanity-check the saved model

Reload it and predict on a few test images to confirm it works before going live.

In [6]:
clf2 = joblib.load("digit_model.joblib")
sample = X_te[:5]
preds = clf2.predict(sample)
print("Predictions:", preds.tolist())
print("Actual     :", y_te[:5].tolist())
print("Model loads and predicts correctly." if list(preds) == list(y_te[:5])
      else "Predictions differ - model still works, just imperfect on these samples.")

Predictions: [8, 4, 8, 7, 7]
Actual     : [8, 4, 8, 7, 7]
Model loads and predicts correctly.


## 4. Write the live air-writing app

This writes `air_writer.py`. Because it shows a camera window, run it from the
terminal (not cell-by-cell):

```
python air_writer.py
```

### How to use it
- **Index finger pointing** -> draw the digit (a trail shows on the camera view)
- **Open palm** -> RECOGNISE what you drew (it predicts and types the digit)
- **Fist** -> CLEAR the canvas and start over
- **`q`** -> quit

The recognised digit is typed wherever your cursor focus is, and also printed
in the camera window.

In [7]:
script = r"""
# air_writer.py - draw a digit in the air, recognise it, type it.
# Uses MediaPipe (hand tracking) + scikit-learn (recognition) + pyautogui (typing).
import time, platform, threading
from collections import deque

import cv2
import numpy as np
import mediapipe as mp
import joblib
import pyautogui

# ---- load the trained model -------------------------------------------------
try:
    MODEL = joblib.load("digit_model.joblib")
except Exception as e:
    raise SystemExit("Could not load digit_model.joblib - run the training "
                     "notebook first. Error: " + str(e))

# ---- landmark constants -----------------------------------------------------
WRIST = 0
THUMB_TIP, INDEX_TIP, MIDDLE_TIP = 4, 8, 12
TIPS = [4, 8, 12, 16, 20]
PIPS = [3, 6, 10, 14, 18]

def fingers_up(lm):
    f = [1 if lm[THUMB_TIP].x < lm[THUMB_TIP - 1].x else 0]
    for tip, pip in zip(TIPS[1:], PIPS[1:]):
        f.append(1 if lm[tip].y < lm[pip].y else 0)
    return f

def preprocess(canvas):
    # canvas: white strokes on black, any size. Convert to MNIST-style 28x28.
    ys, xs = np.where(canvas > 0)
    if len(xs) == 0:
        return None
    # crop to the bounding box of the drawing
    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    crop = canvas[y0:y1+1, x0:x1+1]
    # pad to square
    h, w = crop.shape
    side = max(h, w)
    sq = np.zeros((side, side), np.uint8)
    sq[(side-h)//2:(side-h)//2+h, (side-w)//2:(side-w)//2+w] = crop
    # resize to 20x20 then pad to 28x28 (MNIST digits sit in a 20x20 box)
    small = cv2.resize(sq, (20, 20), interpolation=cv2.INTER_AREA)
    img = np.zeros((28, 28), np.uint8)
    img[4:24, 4:24] = small
    return (img.astype(np.float32) / 255.0).reshape(1, 784)

def main():
    if platform.system() == "Windows":
        cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    else:
        cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    hands = mp.solutions.hands.Hands(static_image_mode=False, max_num_hands=1,
                                     min_detection_confidence=0.7,
                                     min_tracking_confidence=0.5)
    draw_util = mp.solutions.drawing_utils

    canvas = None              # drawing buffer (same size as frame)
    prev_pt = None
    last_pred = None
    cooldown = 0               # frames to wait after a recognise

    print("Draw with index finger. Open palm = recognise. Fist = clear. q = quit.")
    while True:
        ok, frame = cap.read()
        if not ok:
            continue
        frame = cv2.flip(frame, 1)
        h, w = frame.shape[:2]
        if canvas is None:
            canvas = np.zeros((h, w), np.uint8)

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        res = hands.process(rgb)
        rgb.flags.writeable = True

        status = "show hand"
        if res.multi_hand_landmarks:
            hand = res.multi_hand_landmarks[0]
            draw_util.draw_landmarks(frame, hand, mp.solutions.hands.HAND_CONNECTIONS)
            lm = hand.landmark
            f = fingers_up(lm)
            total = sum(f)

            if cooldown > 0:
                cooldown -= 1

            if total == 0:                      # fist -> clear
                canvas[:] = 0
                prev_pt = None
                status = "CLEARED"
            elif total == 5 and cooldown == 0:  # open palm -> recognise
                vec = preprocess(canvas)
                if vec is not None:
                    pred = int(MODEL.predict(vec)[0])
                    last_pred = pred
                    pyautogui.typewrite(str(pred))
                    canvas[:] = 0
                    prev_pt = None
                    cooldown = 20
                    status = f"TYPED: {pred}"
                else:
                    status = "nothing drawn"
            else:
                index_only = f[1] == 1 and f[2] == 0
                if index_only:                  # draw
                    cx = int(lm[INDEX_TIP].x * w)
                    cy = int(lm[INDEX_TIP].y * h)
                    if prev_pt is not None:
                        cv2.line(canvas, prev_pt, (cx, cy), 255, 18)
                    prev_pt = (cx, cy)
                    status = "DRAWING"
                else:
                    prev_pt = None
                    status = "pen up"

        # overlay the canvas strokes in colour on the frame
        colour = cv2.cvtColor(canvas, cv2.COLOR_GRAY2BGR)
        colour[:, :, 0] = 0      # make strokes cyan-ish
        frame = cv2.addWeighted(frame, 1.0, colour, 0.8, 0)

        cv2.putText(frame, status, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        if last_pred is not None:
            cv2.putText(frame, f"last: {last_pred}", (10, 65),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

        cv2.imshow("Air Writer - digit recognition", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()
    hands.close()

if __name__ == "__main__":
    main()
"""
with open("air_writer.py", "w") as fh:
    fh.write(script)
print("Wrote air_writer.py - run it with:  python air_writer.py")

Wrote air_writer.py - run it with:  python air_writer.py


## 5. Run it

```
python air_writer.py
```

Draw a big clear digit with your index finger, then open your palm to recognise
and type it. Make a fist to clear and try again.

### Tuning
- **Wrong predictions?** Draw bigger and more centred. The model expects a
  single digit filling most of the drawing area.
- **Stroke too thin/thick?** Change the `18` in `cv2.line(..., 255, 18)`.
- **Double-types?** Increase the `cooldown = 20` value.
- **Want letters too?** Swap MNIST for EMNIST in training and widen the model -
  ask and I can adapt it.

### What you've built
The complete original vision: a touchless system that tracks your hand, moves
and clicks like a mouse, draws on your desktop, and now reads what you write in
the air and types it - all from a webcam, no special hardware.